# Análisis System Usability Scale (SUS) — SGB-SaaS

## Qué es el SUS

El **System Usability Scale** (Brooke, 1996) es un cuestionario estándar de usabilidad con **10 ítems** en escala Likert **1–5** (1 = totalmente en desacuerdo, 5 = totalmente de acuerdo). Los ítems impares son afirmaciones positivas y los pares negativas. La puntuación agregada por participante se normaliza a un rango **0–100**.

Se usa aquí porque es el instrumento exigido por la guía (Bloque C.3 / RQ3) y porque permite comparar el resultado futuro contra rangos adjetivos publicados (Bangor et al.), sin depender de un instrumento ad hoc.

## Estado real del proyecto (verificado)

| Hecho | Evidencia |
|---|---|
| **N = 0** participantes | `docs/mediciones/DATA-DICTIONARY.md` (alcance: la encuesta SUS no se documenta porque no existe archivo real); `docs/capitulos/08-resultados.tex` §Usabilidad |
| Bloqueado por despliegue público | `OBS-08` en `docs/observaciones/OBSERVACIONES.md`; Cap. Trabajo futuro |
| Target de muestra | $N \geq 15$ (guía de Entrega Final) |
| Protocolo listo | `docs/etica/consentimientos/plantilla.md`; anexo SUS en `docs/capitulos/14-anexos.tex` |

**Este cuaderno es un esqueleto ejecutable.** No inventa respuestas, medias ni scores. Cuando exista el CSV real, la celda de carga dejará de fallar y el resto del pipeline podrá correrse.

## Ruta de datos (verificación)

`docs/mediciones/DATA-DICTIONARY.md` **no define aún** un archivo ni un esquema de campos SUS (cita textual: *"no se documenta aquí ningún campo de esa medición porque todavía no existe ningún archivo real"*).

`docs/mediciones/README.md` sí anticipa la subcarpeta `docs/mediciones/sus/` cuando se ejecute la encuesta.

**Ruta esperada adoptada por este cuaderno** (convención de proyecto, pendiente de materializar y de documentar en el diccionario):

`../docs/mediciones/sus/respuestas.csv` (relativo a `scripts/`)

Esquema CSV esperado cuando existan datos (no hay filas hoy):

`participante_id,item_1,item_2,item_3,item_4,item_5,item_6,item_7,item_8,item_9,item_10`

donde `item_k` ∈ {1,2,3,4,5} y `participante_id` es el código anónimo (`P01`, `P02`, …), nunca nombre ni correo.

## 1. Dependencias y resolución de rutas

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ITEM_COLS = [f"item_{i}" for i in range(1, 11)]

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "sus-analysis.ipynb").exists() or NOTEBOOK_DIR.name == "scripts":
    PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scripts" else NOTEBOOK_DIR.parent
    SUS_CSV = (NOTEBOOK_DIR / "../docs/mediciones/sus/respuestas.csv").resolve()
elif (NOTEBOOK_DIR / "docs" / "mediciones").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
    SUS_CSV = PROJECT_ROOT / "docs" / "mediciones" / "sus" / "respuestas.csv"
else:
    raise FileNotFoundError(
        f"No se pudo resolver la raíz del repo desde CWD={NOTEBOOK_DIR}. "
        "Abrí el cuaderno desde scripts/ o desde la raíz de sgb-saas."
    )

if NOTEBOOK_DIR.name == "scripts":
    PROJECT_ROOT = NOTEBOOK_DIR.parent

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Ruta esperada SUS:", SUS_CSV)
print("¿Existe el archivo?", SUS_CSV.is_file())

## 2. Carga de datos

Si el CSV no existe, esta celda **falla de forma explícita** con el mensaje de N=0. No hay `try/except` silencioso ni datos de relleno.

In [ ]:
DATOS_DISPONIBLES = False
df = None

if not SUS_CSV.is_file():
    msg = (
        "[AVISO] Datos de SUS aún no disponibles — N=0.\n"
        f"Archivo esperado (aún inexistente): {SUS_CSV}\n"
        "Confirmado en docs/mediciones/DATA-DICTIONARY.md: la encuesta SUS "
        "no se ha ejecutado; el diccionario no documenta campos SUS porque "
        "no hay archivo real que los respalde.\n"
        "Este cuaderno NO inventa participantes ni scores. "
        "Cuando exista docs/mediciones/sus/respuestas.csv, re-ejecutá esta celda."
    )
    print(msg)
    raise FileNotFoundError(msg)

df = pd.read_csv(SUS_CSV)
faltan = [c for c in ["participante_id", *ITEM_COLS] if c not in df.columns]
if faltan:
    raise ValueError(
        f"El CSV existe pero le faltan columnas esperadas: {faltan}. "
        "No se inventan columnas ni valores."
    )

n = len(df)
if n == 0:
    raise ValueError(
        "[AVISO] El CSV existe pero está vacío (0 filas) — N=0. "
        "No se fabrican filas de participantes."
    )

DATOS_DISPONIBLES = True
print(f"Datos SUS cargados: N={n} participantes desde {SUS_CSV}")
df.head()


## 3. Cálculo del score SUS (fórmula de Brooke)

Para cada participante:

- Ítems **impares** (1,3,5,7,9): contribución = `puntuación − 1`
- Ítems **pares** (2,4,6,8,10): contribución = `5 − puntuación`
- Score = `(suma de las 10 contribuciones) × 2.5` → rango 0–100

La función queda definida siempre; solo se aplica a un DataFrame cuando `DATOS_DISPONIBLES` es verdadero.

In [ ]:
def score_sus_fila(row: pd.Series) -> float:
    """Score SUS 0–100 para una fila con columnas item_1..item_10 (Likert 1–5)."""
    total = 0.0
    for i in range(1, 11):
        v = float(row[f"item_{i}"])
        if v < 1 or v > 5:
            raise ValueError(
                f"Ítem {i} fuera de Likert 1–5: {v} (participante={row.get('participante_id')})"
            )
        if i % 2 == 1:
            total += v - 1
        else:
            total += 5 - v
    return total * 2.5


def agregar_scores_sus(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["sus_score"] = out.apply(score_sus_fila, axis=1)
    return out


print("Función score_sus_fila / agregar_scores_sus listas.")
if not DATOS_DISPONIBLES:
    print("Sin datos (N=0): no se calculan scores. Saltá a la sección de interpretación (markdown).")
else:
    df = agregar_scores_sus(df)
    print("Scores SUS calculados para N=", len(df))
    df[["participante_id", "sus_score"]].head()

## 4. Estadística descriptiva

Código listo para media, mediana y desviación estándar del score agregado y de cada ítem. **No se imprimen cifras inventadas** si N=0.

In [ ]:
def descriptivos_sus(frame: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    score = frame["sus_score"]
    resumen_score = pd.Series(
        {
            "N": int(score.shape[0]),
            "media": float(score.mean()),
            "mediana": float(score.median()),
            "desv_estandar": float(score.std(ddof=1)) if score.shape[0] > 1 else float("nan"),
        }
    )
    por_item = frame[ITEM_COLS].agg(["mean", "median", "std"]).T
    por_item.columns = ["media", "mediana", "desv_estandar"]
    return resumen_score, por_item


if not DATOS_DISPONIBLES:
    print(
        "[AVISO] Datos de SUS aún no disponibles — N=0. "
        "No se reporta media/mediana/σ inventadas. "
        "Re-ejecutar tras crear docs/mediciones/sus/respuestas.csv."
    )
else:
    resumen_score, por_item = descriptivos_sus(df)
    print("=== Score SUS agregado ===")
    print(resumen_score.round(2))
    print("\n=== Por ítem (Likert 1–5) ===")
    print(por_item.round(2))


## 5. Interpretación — escala adjetiva de Bangor et al.

Tabla de referencia metodológica (Bangor, Kortum & Miller) para interpretar el **score SUS agregado 0–100** cuando existan datos reales. **No es un resultado de este proyecto** (hoy N=0).

| Rango aproximado de score SUS | Adjetivo |
|---|---|
| 0 – ~25 | Peor imaginable |
| ~25 – ~50 | Pobre |
| ~50 – ~70 | OK |
| ~70 – ~80 | Bueno |
| ~80 – ~90 | Excelente |
| ~90 – 100 | Mejor imaginable |

Nota: los cortes exactos varían ligeramente según la publicación/figura usada; al cerrar el Bloque C.3 conviene citar la referencia concreta en el informe y aplicar el mismo criterio a la media (o mediana) real observada — no a un número inventado aquí.

In [ ]:
def adjetivo_bangor(score: float) -> str:
    """Mapeo de referencia (aprox.) score SUS → adjetivo Bangor et al."""
    if score < 25:
        return "Peor imaginable"
    if score < 50:
        return "Pobre"
    if score < 70:
        return "OK"
    if score < 80:
        return "Bueno"
    if score < 90:
        return "Excelente"
    return "Mejor imaginable"


print("Función adjetivo_bangor lista.")
if not DATOS_DISPONIBLES:
    print("N=0: no se asigna adjetivo a ningún score (no hay score).")
else:
    media = float(df["sus_score"].mean())
    print(f"Media SUS = {media:.2f} → {adjetivo_bangor(media)}")

## 6. Visualizaciones (código listo)

1. Barras por ítem: media ± desviación estándar (Likert 1–5).
2. Gauge / barra del score agregado medio contra bandas Bangor.

Sin datos, las celdas solo declaran N=0 y no dibujan cifras ficticias.

In [ ]:
def plot_items_media_sd(frame: pd.DataFrame, ax=None):
    medios = frame[ITEM_COLS].mean()
    desv = frame[ITEM_COLS].std(ddof=1)
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(1, 11)
    ax.bar(x, medios.values, yerr=desv.values, capsize=4, color="#0072B2", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"I{i}" for i in x])
    ax.set_ylim(1, 5)
    ax.set_ylabel("Likert (1–5)")
    ax.set_xlabel("Ítem SUS")
    ax.set_title("Media ± DE por ítem SUS")
    return ax


def plot_score_vs_bangor(score_medio: float, ax=None):
    """Barra horizontal del score medio sobre bandas adjetivas Bangor."""
    bandas = [
        (0, 25, "Peor imaginable", "#D55E00"),
        (25, 50, "Pobre", "#E69F00"),
        (50, 70, "OK", "#F0E442"),
        (70, 80, "Bueno", "#009E73"),
        (80, 90, "Excelente", "#56B4E9"),
        (90, 100, "Mejor imaginable", "#0072B2"),
    ]
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 2.8))
    for lo, hi, label, color in bandas:
        ax.barh(0, hi - lo, left=lo, height=0.6, color=color, alpha=0.55, label=label)
    ax.plot([score_medio], [0], marker="o", markersize=14, color="black", zorder=5)
    ax.axvline(score_medio, color="black", linestyle="--", linewidth=1)
    ax.set_xlim(0, 100)
    ax.set_yticks([])
    ax.set_xlabel("Score SUS (0–100)")
    ax.set_title(f"Score medio = {score_medio:.1f} ({adjetivo_bangor(score_medio)})")
    ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=False)
    return ax


if not DATOS_DISPONIBLES:
    print(
        "[AVISO] Datos de SUS aún no disponibles — N=0. "
        "Gráficos no renderizados (no hay medias reales que dibujar)."
    )
else:
    fig, axes = plt.subplots(2, 1, figsize=(9, 7), gridspec_kw={"height_ratios": [2, 1]})
    plot_items_media_sd(df, ax=axes[0])
    plot_score_vs_bangor(float(df["sus_score"].mean()), ax=axes[1])
    fig.tight_layout()
    plt.show()


## 7. Checklist para cuando lleguen los datos

1. Crear `docs/mediciones/sus/respuestas.csv` con el esquema declarado arriba (solo códigos `Pxx`, Likert 1–5).
2. Ampliar `docs/mediciones/DATA-DICTIONARY.md` con la sección SUS (campos reales).
3. Re-ejecutar este cuaderno de arriba abajo.
4. Trasladar media/mediana/σ y figuras al Cap. Resultados — **sin** inventar valores previos.

Hasta entonces: **N=0**, protocolo listo, análisis pendiente.